In [12]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from ipywidgets import interact, FloatLogSlider, Dropdown, IntSlider

In [13]:
# Load the feature datasets created from the forecasting pipeline
train = pd.read_csv("../data/processed/train_features.csv")
val = pd.read_csv("../data/processed/val_features.csv")

# Convert hour column to datetime for plotting
train["hour"] = pd.to_datetime(train["hour"], utc=True, errors="coerce")
val["hour"] = pd.to_datetime(val["hour"], utc=True, errors="coerce")

# Define the predictor columns used in the regression-family models
feature_cols = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "lag_1",
    "lag_2",
    "lag_24",
    "lag_168",
    "rolling_mean_24",
]

# Define the forecasting target
target_col = "session_count"

X_train_raw = train[feature_cols]
y_train = train[target_col]

X_val_raw = val[feature_cols]
y_val = val[target_col]

print("Train shape:", train.shape)
print("Validation shape:", val.shape)
print("Number of features:", len(feature_cols))

Train shape: (20997, 12)
Validation shape: (1296, 12)
Number of features: 9


In [14]:
def interactive_regularized_regression(model_type="Ridge", alpha=1.0, n_points=200):
    """
    Interactive tuning demo for Ridge and Lasso regression.

    model_type:
        Choose Ridge or Lasso

    alpha:
        Main regularization tuning knob

    n_points:
        Presentation knob controlling how many time points appear in the forecast plot
    """
    # Standardize the features so regularization acts fairly across all inputs
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)

    # Create the selected model
    if model_type == "Ridge":
        model = Ridge(alpha=alpha)
    else:
        model = Lasso(alpha=alpha, max_iter=10000)

    # Fit the model on the training data
    model.fit(X_train, y_train)

    # Predict on the validation set
    val_pred = model.predict(X_val)

    # Compute validation metrics
    rmse = mean_squared_error(y_val, val_pred) ** 0.5
    r2 = r2_score(y_val, val_pred)

    print(f"Model: {model_type}")
    print(f"Alpha: {alpha}")
    print(f"Validation RMSE: {rmse:.4f}")
    print(f"Validation R²: {r2:.4f}")

    # Prepare the plotting table
    plot_df = val.copy()
    plot_df["actual"] = y_val.values
    plot_df["predicted"] = val_pred
    plot_df = plot_df.iloc[:n_points]

    # Create two vertically stacked plots
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))

    # Plot 1: actual vs predicted forecast
    axes[0].plot(plot_df["hour"], plot_df["actual"], label="Actual")
    axes[0].plot(plot_df["hour"], plot_df["predicted"], label="Predicted")
    axes[0].set_title(f"{model_type}: Actual vs Predicted (Validation, Standardized Features)")
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel("Session Count")
    axes[0].legend()

    # Plot 2: learned coefficients
    coef_df = pd.DataFrame({
        "feature": feature_cols,
        "coefficient": model.coef_,
    }).sort_values("coefficient")

    axes[1].barh(coef_df["feature"], coef_df["coefficient"])
    axes[1].set_title(f"{model_type}: Coefficients (Standardized Features)")
    axes[1].set_xlabel("Coefficient Value")
    axes[1].set_ylabel("Feature")

    plt.tight_layout()
    plt.show()

In [ ]:
interact(
    interactive_regularized_regression,
    model_type=Dropdown(
        options=["Ridge", "Lasso"],
        value="Ridge",
        description="Model"
    ),
    alpha=FloatLogSlider(
        value=1.0,
        base=10,
        min=-4,
        max=2,
        step=0.1,
        description="Alpha"
    ),
    n_points=IntSlider(
        value=200,
        min=50,
        max=500,
        step=50,
        description="Points"
    ),
);

interactive(children=(Dropdown(description='Model', options=('Ridge', 'Lasso'), value='Ridge'), FloatLogSlider…